In [ ]:
import pandas as pd
import numpy as np
import joblib
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

print("Scikit-learn version:", sklearn.__version__)

Scikit-learn version: 1.8.0


In [ ]:
df = pd.read_csv("India Agriculture Crop Production.csv")

print("Initial Shape:", df.shape)
df.head()

Initial Shape: (345407, 10)


,State,District,Crop,Year,Season,Area,Area Units,Production,Production Units,Yield
0,Andaman and Nicobar Islands,NICOBARS,Arecanut,2001-02,Kharif,1254.0,Hectare,2061.0,Tonnes,1.643541
1,Andaman and Nicobar Islands,NICOBARS,Arecanut,2002-03,Whole Year,1258.0,Hectare,2083.0,Tonnes,1.655803
2,Andaman and Nicobar Islands,NICOBARS,Arecanut,2003-04,Whole Year,1261.0,Hectare,1525.0,Tonnes,1.209358
3,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,Arecanut,2001-02,Kharif,3100.0,Hectare,5239.0,Tonnes,1.690000
4,Andaman and Nicobar Islands,SOUTH ANDAMANS,Arecanut,2002-03,Whole Year,3105.0,Hectare,5267.0,Tonnes,1.696296


In [ ]:
# Remove missing values
df = df.dropna(subset=["Production", "Area"])

# Keep only Kerala
df = df[df["State"] == "Kerala"].copy()

# Keep only Tonnes production (important!)
df = df[df["Production Units"] == "Tonnes"].copy()

# Remove zero area
df = df[df["Area"] > 0]

print("After cleaning:", df.shape)

After cleaning: (4303, 10)


In [ ]:
if df["Year"].dtype == "object":
    df["Year"] = df["Year"].str.split("-").str[0]

df["Year"] = df["Year"].astype(int)

In [ ]:
df["yield_per_hectare"] = df["Production"] / df["Area"]

# Remove unrealistic outliers
df = df[df["yield_per_hectare"] < 100]

print("Yield statistics:")
print(df["yield_per_hectare"].describe())

Yield statistics:
count    4234.000000
mean        7.017043
std        13.908831
min         0.000000
25%         0.611652
50%         2.144245
75%         6.198863
max        97.500000
Name: yield_per_hectare, dtype: float64


In [ ]:
features = ["Crop", "District", "Season", "Year"]
target = "yield_per_hectare"

df = df[features + [target]]

In [ ]:
le_crop = LabelEncoder()
le_district = LabelEncoder()
le_season = LabelEncoder()

df["Crop"] = le_crop.fit_transform(df["Crop"])
df["District"] = le_district.fit_transform(df["District"])
df["Season"] = le_season.fit_transform(df["Season"])

print("Encoding complete")

Encoding complete


In [ ]:
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [ ]:
y_pred = model.predict(X_test)

print("Train R2:", model.score(X_train, y_train))
print("Test R2:", model.score(X_test, y_test))
print("MAE:", mean_absolute_error(y_test, y_pred))

Train R2: 0.9670046294564771
Test R2: 0.9066930422228556
MAE: 1.000721187605794


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
joblib.dump(model, "kerala_yield_model.pkl")
joblib.dump(le_crop, "le_crop.pkl")
joblib.dump(le_district, "le_district.pkl")
joblib.dump(le_season, "le_season.pkl")

print("Model and encoders saved successfully ✅")

Model and encoders saved successfully ✅


In [ ]:
print(df["yield_per_hectare"].describe())

count    4234.000000
mean        7.017043
std        13.908831
min         0.000000
25%         0.611652
50%         2.144245
75%         6.198863
max        97.500000
Name: yield_per_hectare, dtype: float64


In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("Cross Validation R2 Scores:", cv_scores)
print("Average CV R2:", cv_scores.mean())

Cross Validation R2 Scores: [0.94525156 0.82242873 0.84759118 0.92199056 0.90307287]
Average CV R2: 0.8880669785235117
